In [ ]:
from pathlib import Path

import torch
import torchvision

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

input_dir = Path("/kaggle/input")

print("\nInput folders:")
for path in sorted(input_dir.iterdir()):
    print(path)

In [ ]:
from pathlib import Path

import pandas as pd

articles_path = next(
    Path("/kaggle/input/competitions").glob("*/articles.csv")
)
data_dir = articles_path.parent

articles = pd.read_csv(
    articles_path,
    dtype={"article_id": str, "product_code": str},
)

articles["article_id"] = articles["article_id"].str.zfill(10)

image_paths = sorted((data_dir / "images").rglob("*.jpg"))

image_table = pd.DataFrame({
    "article_id": [path.stem for path in image_paths],
    "image_path": [str(path) for path in image_paths],
})

manifest = articles.merge(
    image_table,
    on="article_id",
    how="inner",
    validate="one_to_one",
)

print("Dataset folder:", data_dir)
print("Articles:", len(articles))
print("Image files:", len(image_table))
print("Matched articles:", len(manifest))
print("Articles without images:", len(articles) - len(manifest))

display(manifest[
    ["article_id", "product_code", "product_type_name",
     "colour_group_name", "image_path"]
].head())

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

sample = manifest.sample(n=6, random_state=42)

fig, axes = plt.subplots(2, 3, figsize=(12, 9))

for ax, (_, row) in zip(axes.flat, sample.iterrows()):
    with Image.open(row["image_path"]) as image:
        ax.imshow(image.convert("RGB"))

    ax.set_title(
        f"{row['product_type_name']}\n"
        f"{row['colour_group_name']} | {row['article_id']}"
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

SEED = 42

group_sizes = manifest.groupby("product_code").size()
eligible_groups = group_sizes[group_sizes >= 2]

selected_codes = eligible_groups.sample(
    n=len(eligible_groups),
    random_state=SEED,
).index.to_numpy()

train_codes, remaining_codes = train_test_split(
    selected_codes,
    test_size=0.20,
    random_state=SEED,
)

val_codes, test_codes = train_test_split(
    remaining_codes,
    test_size=0.50,
    random_state=SEED,
)

split_groups = {
    "train": train_codes,
    "validation": val_codes,
    "test": test_codes,
}

parts = []

for split_name, codes in split_groups.items():
    part = manifest[
        manifest["product_code"].isin(codes)
    ].copy()

    part["split"] = split_name
    parts.append(part)

subset = pd.concat(parts, ignore_index=True)

assert subset.groupby("product_code")["split"].nunique().max() == 1

WORK_DIR = Path("/kaggle/working/hm_visual")
WORK_DIR.mkdir(parents=True, exist_ok=True)

subset.to_csv(WORK_DIR / "subset_manifest.csv", index=False)

print("All matched product groups:", len(group_sizes))
print("Groups with at least two images:", len(eligible_groups))

display(
    subset.groupby("split").agg(
        images=("article_id", "size"),
        product_groups=("product_code", "nunique"),
    )
)

In [ ]:
from torchvision import transforms

train_df = subset[subset["split"] == "train"].reset_index(drop=True)
val_df = subset[subset["split"] == "validation"].reset_index(drop=True)

class_names = sorted(
    train_df["product_group_name"]
    .value_counts()
    .head(4)
    .index
    .tolist()
)

warmup_train = train_df[
    train_df["product_group_name"].isin(class_names)
].reset_index(drop=True)

warmup_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

row = warmup_train.iloc[0]

with Image.open(row["image_path"]) as image:
    image_tensor = warmup_transform(image.convert("RGB"))

print("Classes:", class_names)
print("Image tensor shape:", image_tensor.shape)
print("Data type:", image_tensor.dtype)
print("Value range:", image_tensor.min().item(), image_tensor.max().item())

In [ ]:
import torch
from torch import nn

class SmallCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(32, num_classes)

    def forward(self, images):
        features = self.features(images)
        features = torch.flatten(features, start_dim=1)
        logits = self.classifier(features)   # ✅ düzeltildi
        return logits


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SmallCNN(num_classes=len(class_names)).to(device)
model.eval()

image_batch = image_tensor.unsqueeze(0).to(device)

with torch.no_grad():
    logits = model(image_batch)

print("Batch shape:", image_batch.shape)
print("Output shape:", logits.shape)

In [ ]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader


class FashionDataset(Dataset):
    def __init__(self, frame, label_to_id, transform):
        self.frame = frame.reset_index(drop=True)
        self.label_to_id = label_to_id
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]

        with Image.open(row["image_path"]) as image:
            image_tensor = self.transform(image.convert("RGB"))

        label = self.label_to_id[row["product_group_name"]]

        return image_tensor, label


label_to_id = {
    name: index
    for index, name in enumerate(class_names)
}

train_dataset = FashionDataset(
    warmup_train,
    label_to_id,
    warmup_transform,
)

image, label = train_dataset[0]

print("Category numbers:", label_to_id)
print("First image:", image.shape)
print("Correct category:", class_names[label])

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
)

images, labels = next(iter(train_loader))

print("Image batch:", images.shape)
print("Label batch:", labels.shape)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
from torch import nn

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

model = SmallCNN(num_classes=len(class_names)).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

warmup_val = val_df[
    val_df["product_group_name"].isin(class_names)
].reset_index(drop=True)

val_dataset = FashionDataset(
    warmup_val,
    label_to_id,
    warmup_transform,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2,
)

majority_class = warmup_train["product_group_name"].value_counts().idxmax()
baseline_accuracy = (
    warmup_val["product_group_name"] == majority_class
).mean()

print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Always predicting:", majority_class)
print(f"Majority baseline accuracy: {baseline_accuracy:.4f}")

In [ ]:
from time import perf_counter

from tqdm.auto import tqdm

EPOCHS = 2
history = []
best_f1 = -1.0

for epoch in range(EPOCHS):
    started = perf_counter()
    model.train()

    total_loss = 0.0
    total_images = 0

    for images, labels in tqdm(
        train_loader,
        desc=f"Training epoch {epoch + 1}/{EPOCHS}",
    ):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        logits = model(images)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        total_images += images.size(0)

    train_loss = total_loss / total_images

    model.eval()
    predictions = []
    targets = []

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="Validation"):
            images = images.to(device)

            logits = model(images)
            predicted = logits.argmax(dim=1)

            predictions.extend(predicted.cpu().tolist())
            targets.extend(labels.tolist())

    val_accuracy = accuracy_score(targets, predictions)
    val_f1 = f1_score(
        targets,
        predictions,
        labels=list(range(len(class_names))),
        average="macro",
        zero_division=0,
    )

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_accuracy": val_accuracy,
        "val_macro_f1": val_f1,
        "seconds": perf_counter() - started,
    })

    pd.DataFrame(history).to_csv(
        WORK_DIR / "small_cnn_history.csv",
        index=False,
    )

    if val_f1 > best_f1:
        best_f1 = val_f1

        torch.save({
            "model_state_dict": model.state_dict(),
            "class_names": class_names,
            "image_size": 128,
            "seed": SEED,
            "val_macro_f1": val_f1,
        }, WORK_DIR / "small_cnn_best.pt")

    print(
        f"Epoch {epoch + 1} | "
        f"loss={train_loss:.4f} | "
        f"val_accuracy={val_accuracy:.4f} | "
        f"val_macro_f1={val_f1:.4f}"
    )

display(pd.DataFrame(history))

In [ ]:
print("CUDA available:", torch.cuda.is_available())
print("Selected device:", device)
print("Model device:", next(model.parameters()).device)